# 01 - Ingesta Parquet a RAW (Yellow + Green, 2015-2025)

Objetivos:
- Leer Parquet publicos de NYC TLC mes a mes.
- Estandarizar tipos/timestamps minimos.
- Escribir a `RAW.YELLOW_TRIPS` / `RAW.GREEN_TRIPS` con metadatos de lineage.
- Registrar auditoria por lote en `RAW.LOAD_AUDIT`.
- **Idempotencia**: reingestar un mes no duplica filas (DELETE WHERE source_year/month antes de append).

Todos los parametros provienen de variables de ambiente (`.env`).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import datetime
import os


In [ ]:
# Parametros desde variables de ambiente (.env)
services = [s.strip() for s in os.environ.get('SERVICES', 'yellow,green').replace(';', ',').split(',') if s.strip()]
start_year = int(os.environ.get('START_YEAR', '2015'))
end_year = int(os.environ.get('END_YEAR', '2025'))
months = [int(m) for m in os.environ.get('MONTHS', '1,2,3,4,5,6,7,8,9,10,11,12').split(',') if m.strip()]
run_id = os.environ.get('RUN_ID') or datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

yellow_base = os.environ.get('YELLOW_BASE_URL', 'https://d37ci6vzurychx.cloudfront.net/trip-data')
green_base = os.environ.get('GREEN_BASE_URL', 'https://d37ci6vzurychx.cloudfront.net/trip-data')

print(f'services={services} years={start_year}-{end_year} months={months} run_id={run_id}')


In [ ]:
app = (
    SparkSession.builder
    .appName('01_ingesta_parquet_raw')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

# Credenciales Snowflake desde variables de ambiente
sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'RAW'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

# Helper para correr SQL arbitrario en Snowflake via JVM del conector spark-snowflake
sf_utils = app._jvm.net.snowflake.spark.snowflake.Utils

def sf_run(sql):
    jsf = app._jvm.java.util.HashMap()
    for k, v in sf_option.items():
        jsf.put(k, v)
    sf_utils.runQuery(jsf, sql)


In [ ]:
# Asegurar que existan los esquemas antes de cargar
for schema in [os.environ.get('SF_RAW_SCHEMA', 'RAW'), 'CURATED', os.environ.get('SF_ANALYTICS_SCHEMA', 'ANALYTICS')]:
    sf_run(f'CREATE SCHEMA IF NOT EXISTS {schema}')
print('Schemas verificados.')


In [ ]:
def get_nyc_url(service, year, month):
    base = yellow_base if service == 'yellow' else green_base
    return f'{base}/{service}_tripdata_{year}-{month:02d}.parquet'


In [ ]:
def compute_trip_nk(df, service):
    """Hash SHA-256 sobre columnas clave. Usa tpep_* para yellow y lpep_* para green."""
    if service == 'yellow':
        pu_col, do_col = 'tpep_pickup_datetime', 'tpep_dropoff_datetime'
    else:
        pu_col, do_col = 'lpep_pickup_datetime', 'lpep_dropoff_datetime'
    key_cols = [F.lit(service), F.col('VendorID').cast('string'),
                F.col(pu_col).cast('string'), F.col(do_col).cast('string'),
                F.col('total_amount').cast('string')]
    return df.withColumn('trip_nk', F.sha2(F.concat_ws('||', *key_cols), 256))


In [ ]:
def log_audit(service, year, month, status, rows_read=0, rows_written=0, notes=''):
    row = [(run_id, service, int(year), int(month), status,
            int(rows_read), int(rows_written),
            datetime.datetime.utcnow(), notes)]
    schema = 'run_id string, service string, year int, month int, status string, rows_read long, rows_written long, event_timestamp timestamp, notes string'
    audit_df = app.createDataFrame(row, schema)
    (audit_df.write.format('snowflake')
        .options(**sf_option)
        .option('dbtable', 'LOAD_AUDIT')
        .mode('append')
        .save())


In [ ]:
# Main loop: descarga, estandariza, idempotencia via DELETE + append, y audita
for service in services:
    target_table = f'{service.upper()}_TRIPS'
    for year in range(start_year, end_year + 1):
        for month in months:
            url = get_nyc_url(service, year, month)
            try:
                df_raw = app.read.parquet(url)
                rows_read = df_raw.count()

                df_processed = (
                    df_raw
                    .withColumn('run_id', F.lit(run_id))
                    .withColumn('service_type', F.lit(service))
                    .withColumn('source_year', F.lit(year).cast('int'))
                    .withColumn('source_month', F.lit(month).cast('int'))
                    .withColumn('ingested_at_utc', F.current_timestamp())
                    .withColumn('source_path', F.lit(url))
                )
                df_processed = compute_trip_nk(df_processed, service)

                # Idempotencia: borrar filas previas del mismo lote antes de append
                sf_raw_schema = os.environ.get('SF_RAW_SCHEMA', 'RAW')
                try:
                    sf_run(f"DELETE FROM {sf_raw_schema}.{target_table} WHERE source_year = {year} AND source_month = {month}")
                except Exception as del_err:
                    # La tabla puede no existir aun en la primera corrida; el append la crea.
                    print(f'  (delete skip {service} {year}-{month}: {str(del_err)[:120]})')

                (df_processed.write.format('snowflake')
                    .options(**sf_option)
                    .option('dbtable', target_table)
                    .mode('append')
                    .save())

                log_audit(service, year, month, 'SUCCESS', rows_read, rows_read)
                print(f'OK  {service} {year}-{month:02d}  rows={rows_read}')
            except Exception as e:
                msg = str(e)
                status = 'Missing' if ('403' in msg or '404' in msg or 'NoSuchKey' in msg) else 'ERROR'
                try:
                    log_audit(service, year, month, status, 0, 0, notes=msg[:500])
                except Exception as le:
                    print(f'  audit log failed: {le}')
                print(f'{status:7s} {service} {year}-{month:02d}  {msg[:160]}')


In [ ]:
# Resumen del run actual
df_audit_summary = (
    app.read.format('snowflake').options(**sf_option)
    .option('dbtable', 'LOAD_AUDIT').load()
    .filter(F.col('run_id') == run_id)
)
df_audit_summary.groupBy('service', 'status').count().orderBy('service', 'status').show()
